# 03. Supervised Segmentation 
The goal of this stage is to **identify which features are most informative with respect to the churn outcome**, using supervised segmentation techniques.

Rather than training a full predictive model, this step focuses on understanding how individual features partition the customer population into groups with different churn behavior.

This stage answers the question:

> Which customer attributes reduce uncertainty about churn the most?

## 3.1. Environment setup

In [1]:
import numpy as np
import pandas as pd
from math import log2

import matplotlib.pyplot as plt
import seaborn as sns

## 3.2. Load prepared training data

In [4]:
X_train = pd.read_csv("../data/X_train.csv")
y_train = pd.read_csv("../data/y_train.csv")

### 3.2.1. Quick sanity checks

In [5]:
print(X_train.shape)
print(y_train.shape)

X_train.head(), y_train.head()

(4800, 17)
(4800, 1)


(   customer_id   plan region    industry  tenure_months  team_size  \
 0         2243  Basic   APAC  Healthcare             31          2   
 1          264  Basic     EU  E-commerce              1          7   
 2         1801    Pro    NaN        SaaS             17          5   
 3         5321  Basic    NaN  E-commerce             39          3   
 4         2076    Pro     EU        SaaS              5          1   
 
    monthly_price  logins_per_week  campaigns_per_month  automation_used  \
 0           9.93             4.17                    6                0   
 1          26.03             5.50                    5                0   
 2          54.52             3.41                    7                1   
 3          16.02             2.56                    5                1   
 4          53.85             1.87                    4                1   
 
    onboarding_completed  integrations_connected  support_tickets_90d  \
 0                     0                 

## 3.3. Base entropy
We first compute the entropy of the target variable (`churn`) on the training set.

This value represents the baseline level of uncertainty about churn **before** considering any features.
All subsequent information gain calculations measure how much individual features reduce this baseline uncertainty.

In [6]:
def entropy(y):
    probs = y.value_counts(normalize=True)
    return -np.sum(probs * np.log2(probs))

base_entropy = entropy(y_train)
base_entropy

np.float64(0.922811132172962)

## 3.4. Segmentation rules
Supervised segmentation is performed to assess feature informativeness prior to any preprocessing or model fitting.

The following rules are applied:

- No imputation is performed; missing values are treated as an explicit category.
- No scaling or normalization is applied, as entropy-based measures depend only on class proportions.
- Categorical features are evaluated in their original form (no one-hot encoding).
- Binary features are segmented directly into two groups (0 / 1).
- Numeric features are discretized using simple, interpretable splits for entropy calculation only.
- All computations are based on training data exclusively.

These choices ensure that entropy and information gain reflect the raw informational structure of the data.